# New Zealand Dataset — Stratified Sampling (Parquet Optimised)

**Goal:** Sample **1B tokens** out of 60.7B tokens, ensuring every CC-MAIN batch directory is covered.

**Data format (confirmed):**
- Each `CC-MAIN-YYYY-WW/` contains `chunk_*.parquet` files
- Fields: `text, id, dump, url, date, file_path, language, language_score, token_count, index`
- **`token_count` is pre-computed**, so we can skip tiktoken — large speed-up.

| Item | Value |
|---|---|
| Total tokens | 60,714,120,289 |
| Total files | 27,168 |
| Target tokens | 1,000,000,000 |
| Base sampling ratio | ~1.647% |

In [1]:
#!pip install pyarrow tqdm ipywidgets

In [2]:
# ── Cell 1: Imports & Configuration ─────────────────────────────
import os, json, random, time
from pathlib import Path
from collections import defaultdict
import pyarrow.parquet as pq
from tqdm.notebook import tqdm

# ════════════════════════════════════════════════════════════════
DATA_ROOT  = Path('/home/jovyan/data/New_Zealand')
OUTPUT_DIR = Path('./nz_sample')
# ════════════════════════════════════════════════════════════════

OUTPUT_FILE = OUTPUT_DIR / 'nz_1B_sample.parquet'
STATS_FILE  = OUTPUT_DIR / 'sampling_stats.json'

TARGET_TOKENS = 1_000_000_000
TOTAL_TOKENS  = 60_714_120_289
SAMPLE_RATIO  = TARGET_TOKENS / TOTAL_TOKENS   # ≈ 1.647%
RANDOM_SEED   = 42

random.seed(RANDOM_SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'DATA_ROOT    : {DATA_ROOT}')
print(f'OUTPUT_FILE  : {OUTPUT_FILE}')
print(f'SAMPLE_RATIO : {SAMPLE_RATIO:.4%}')

DATA_ROOT    : /home/jovyan/data/New_Zealand
OUTPUT_FILE  : nz_sample/nz_1B_sample.parquet
SAMPLE_RATIO : 1.6471%


In [3]:
# ── Cell 2: Scan all CC-MAIN directories and locate .parquet files ──────────
dir_files = {}   # { 'CC-MAIN-2013-20': [Path, Path, ...] }

for batch_dir in sorted(DATA_ROOT.iterdir()):
    if not batch_dir.is_dir() or not batch_dir.name.startswith('CC-MAIN'):
        continue
    files = sorted(batch_dir.glob('*.parquet'))
    if files:
        dir_files[batch_dir.name] = files

all_dirs = sorted(dir_files.keys())
n_dirs   = len(all_dirs)
total_files_scanned = sum(len(v) for v in dir_files.values())

print(f'CC-MAIN directories found : {n_dirs}')
print(f'parquet files scanned     : {total_files_scanned}')
print(f'\nFiles per directory (first 10):')
for d in all_dirs[:10]:
    print(f'  {d}: {len(dir_files[d])} files')
if n_dirs > 10:
    print(f'  ... and {n_dirs-10} more directories')

发现 CC-MAIN 目录数 : 109
扫描到 parquet 文件 : 128

各目录文件数（前 10 条）:
  CC-MAIN-2013-20: 1 个文件
  CC-MAIN-2013-48: 1 个文件
  CC-MAIN-2014-10: 1 个文件
  CC-MAIN-2014-15: 1 个文件
  CC-MAIN-2014-23: 1 个文件
  CC-MAIN-2014-35: 1 个文件
  CC-MAIN-2014-41: 1 个文件
  CC-MAIN-2014-42: 1 个文件
  CC-MAIN-2014-49: 1 个文件
  CC-MAIN-2014-52: 1 个文件
  ... 还有 99 个目录


In [4]:
# ── Cell 3: Read each directory's true token count from progress.json ─────
# More accurate than file-count proportion: allocate quota by token share directly

dir_tokens = {}
for d in all_dirs:
    progress = DATA_ROOT / d / 'progress.json'
    if progress.exists():
        with open(progress) as f:
            info = json.load(f)
        dir_tokens[d] = info.get('total_tokens', 0)
    else:
        dir_tokens[d] = 0   # If no progress.json, estimate via file count later

sum_known_tokens = sum(dir_tokens.values())
print(f'Total tokens collected from progress.json : {sum_known_tokens:,}')
print(f'Total tokens declared in metadata         : {TOTAL_TOKENS:,}')
print(f'\nTop 5 directories by token count:')
for d, t in sorted(dir_tokens.items(), key=lambda x: -x[1])[:5]:
    print(f'  {d}: {t:>14,} tokens')

从 progress.json 收集到的总 tokens: 60,714,120,289
元数据声明的总 tokens          : 60,714,120,289

Token 数最大的 5 个目录:
  CC-MAIN-2023-50:  1,016,025,307 tokens
  CC-MAIN-2022-21:    991,405,053 tokens
  CC-MAIN-2020-40:    877,020,727 tokens
  CC-MAIN-2021-04:    872,987,268 tokens
  CC-MAIN-2021-31:    864,843,806 tokens


In [5]:
# ── Cell 4: Stratified quota calculation (by token share, min 1 file per directory) ─
# Per-directory target tokens = SAMPLE_RATIO × total tokens in that directory
# Files to sample = ceil(directory file count × SAMPLE_RATIO), minimum 1

import math

quota_per_dir = {}
for d in all_dirs:
    n_files = len(dir_files[d])
    # Sample proportionally by file count: at least 1, not more than the actual count
    n_to_pick = max(1, min(math.ceil(n_files * SAMPLE_RATIO), n_files))
    quota_per_dir[d] = n_to_pick

total_selected_files = sum(quota_per_dir.values())
print(f'Files actually selected : {total_selected_files}')
print(f'\nQuota per directory (first 10):')
for d in all_dirs[:10]:
    print(f'  {d}: {len(dir_files[d])} files → sample {quota_per_dir[d]}')

实际选取文件数  : 109

各目录配额（前 10 条）:
  CC-MAIN-2013-20: 1 文件 → 抽 1 个
  CC-MAIN-2013-48: 1 文件 → 抽 1 个
  CC-MAIN-2014-10: 1 文件 → 抽 1 个
  CC-MAIN-2014-15: 1 文件 → 抽 1 个
  CC-MAIN-2014-23: 1 文件 → 抽 1 个
  CC-MAIN-2014-35: 1 文件 → 抽 1 个
  CC-MAIN-2014-41: 1 文件 → 抽 1 个
  CC-MAIN-2014-42: 1 文件 → 抽 1 个
  CC-MAIN-2014-49: 1 文件 → 抽 1 个
  CC-MAIN-2014-52: 1 文件 → 抽 1 个


In [6]:
# ── Cell 5: Randomly pick the specific files ─────────────────────────
selected_files = []
for d in all_dirs:
    pool   = dir_files[d]
    quota  = quota_per_dir[d]
    chosen = random.sample(pool, quota)
    selected_files.extend(chosen)

random.shuffle(selected_files)

print(f'Final files selected : {len(selected_files)}')
print(f'\nSample (first 5 files):')
for f in selected_files[:5]:
    print(f'  {f.parent.name}/{f.name}')

最终选取文件数 : 109

样本（前 5 个文件）:
  CC-MAIN-2016-40/chunk_0.parquet
  CC-MAIN-2021-10/chunk_0.parquet
  CC-MAIN-2018-39/chunk_0.parquet
  CC-MAIN-2015-40/chunk_0.parquet
  CC-MAIN-2024-22/chunk_0.parquet


In [7]:
# ── Cell 6: Main sampling loop (streaming write, low memory) ─────────
import pyarrow as pa
from collections import defaultdict
from tqdm import tqdm

tokens_per_dir_quota = TARGET_TOKENS // n_dirs
print(f"Target contribution per directory: {tokens_per_dir_quota:,} tokens")

files_by_dir = defaultdict(list)
for fp in selected_files:
    files_by_dir[fp.parent.name].append(fp)

total_tokens_written = 0
total_rows_written   = 0
files_done           = 0
t0                   = time.time()
stats_per_dir        = defaultdict(lambda: {'files': 0, 'tokens': 0, 'rows': 0})

pbar = tqdm(total=TARGET_TOKENS, unit='tok', unit_scale=True, desc='Sampling progress', mininterval=0.5)

writer = None   # Streaming ParquetWriter

for dir_name in all_dirs:
    dir_quota = tokens_per_dir_quota
    dir_collected = 0

    for fp in files_by_dir[dir_name]:
        if dir_collected >= dir_quota:
            break
        try:
            table = pq.read_table(fp)
        except Exception as e:
            print(f'[WARN] skipping {fp.name}: {e}')
            continue

        token_counts = table.column('token_count').to_numpy()
        cumsum = token_counts.cumsum()
        need = dir_quota - dir_collected

        if cumsum[-1] <= need:
            take_n = len(token_counts)
            take_tokens = int(cumsum[-1])
        else:
            idx = int((cumsum >= need).argmax()) + 1
            take_n = idx
            take_tokens = int(cumsum[idx-1])
            table = table.slice(0, take_n)

        if writer is None:
            writer = pq.ParquetWriter(OUTPUT_FILE, table.schema, compression='snappy')

        writer.write_table(table)
        del table

        dir_collected        += take_tokens
        total_tokens_written += take_tokens
        total_rows_written   += take_n
        stats_per_dir[dir_name]['files']  += 1
        stats_per_dir[dir_name]['tokens'] += take_tokens
        stats_per_dir[dir_name]['rows']   += take_n
        files_done += 1
        pbar.update(take_tokens)

        if files_done % 5 == 0:
            elapsed = time.time() - t0
            tps = total_tokens_written / max(elapsed, 1e-6)
            print(f"  [{files_done}] {dir_name}: cumulative {total_tokens_written/1e6:.0f}M tokens, covered {len(stats_per_dir)}/{n_dirs} dirs, speed {tps/1e6:.1f}M tok/s")

if writer is not None:
    writer.close()
pbar.close()

elapsed_total = time.time() - t0
print(f'\n{"="*55}')
print(f'✅ Sampling complete!')
print(f'   Total tokens    : {total_tokens_written:,}')
print(f'   Total rows      : {total_rows_written:,}')
print(f'   Files processed : {files_done}')
print(f'   Dirs covered    : {len(stats_per_dir)} / {n_dirs}')
print(f'   Elapsed time    : {elapsed_total/60:.1f} min')
print(f'   Avg speed       : {total_tokens_written/elapsed_total/1e6:.1f}M tok/s')
print(f'   Output file     : {OUTPUT_FILE}')
print(f'   Output size     : {OUTPUT_FILE.stat().st_size/1024/1024:.1f} MB')

每个目录目标贡献: 9,174,311 tokens


抽样进度:   5%|▍         | 45.9M/1.00G [00:36<12:15, 1.30Mtok/s]

  [5] CC-MAIN-2014-23: 累计 46M tokens, 覆盖 5/109 目录, 速度 1.3M tok/s


抽样进度:   9%|▉         | 91.7M/1.00G [01:12<11:47, 1.28Mtok/s]

  [10] CC-MAIN-2014-52: 累计 92M tokens, 覆盖 10/109 目录, 速度 1.3M tok/s


...

  [100] CC-MAIN-2024-38: 累计 918M tokens, 覆盖 100/109 目录, 速度 0.9M tok/s


抽样进度:  96%|█████████▋| 963M/1.00G [18:46<00:56, 647ktok/s]

  [105] CC-MAIN-2025-08: 累计 963M tokens, 覆盖 105/109 目录, 速度 0.9M tok/s


抽样进度: 1.00Gtok [19:18, 863ktok/s]                          


✅ 抽样完成！
   总 tokens     : 1,000,117,633
   总行数        : 1,749,277
   处理文件数    : 109
   覆盖目录数    : 109 / 109
   总耗时        : 19.3 分钟
   平均速度      : 0.9M tok/s
   输出文件      : nz_sample/nz_1B_sample.parquet
   输出大小      : 2835.7 MB


In [8]:
# ── Cell 7: Save the statistics report ───────────────────────────
stats = {
    'dataset': 'New_Zealand',
    'sampling_method': 'stratified_by_cc_main_batch_min1_per_dir',
    'random_seed': RANDOM_SEED,
    'target_tokens': TARGET_TOKENS,
    'total_tokens_written': total_tokens_written,
    'total_rows_written': total_rows_written,
    'files_processed': files_done,
    'dirs_covered': len(stats_per_dir),
    'total_dirs': n_dirs,
    'elapsed_minutes': round(elapsed_total / 60, 2),
    'output_size_mb': round(OUTPUT_FILE.stat().st_size / 1024 / 1024, 1),
    'per_directory': {k: v for k, v in sorted(stats_per_dir.items())}
}

with open(STATS_FILE, 'w') as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)

print(f'Stats report saved: {STATS_FILE}\n')
print(f'{"Directory":<25} {"Files":>5} {"Tokens":>14} {"Rows":>10}')
print('-' * 60)
for d, s in sorted(stats_per_dir.items()):
    print(f'{d:<25} {s["files"]:>5} {s["tokens"]:>14,} {s["rows"]:>10,}')
print('-' * 60)
print(f'{"Total":<25} {files_done:>5} {total_tokens_written:>14,} {total_rows_written:>10,}')

统计报告已保存: nz_sample/sampling_stats.json

目录                           文件         Tokens         行数
------------------------------------------------------------
CC-MAIN-2013-20               1      9,174,413     15,124
CC-MAIN-2013-48               1      9,176,807     13,870
CC-MAIN-2014-10               1      9,174,640     14,007
CC-MAIN-2014-15               1      9,174,581     13,588
CC-MAIN-2014-23               1      9,174,378     13,680
CC-MAIN-2014-35               1      9,174,869     13,795
CC-MAIN-2014-41               1      9,174,723     13,567
CC-MAIN-2014-42               1      9,175,804     13,665
CC-MAIN-2014-49               1      9,174,385     13,807
CC-MAIN-2014-52               1      9,174,637     13,869
CC-MAIN-2015-06               1      9,174,481     13,623
CC-MAIN-2015-11               1      9,174,337     13,488
CC-MAIN-2015-14               1      9,175,124     13,347
CC-MAIN-2015-18               1      9,174,974     13,842
CC-MAIN-2015-22              

In [9]:
# ── Cell 8: Verify the output file ───────────────────────────────
verify = pq.read_table(OUTPUT_FILE)
print(f'Output total rows : {verify.num_rows:,}')
print(f'Output columns    : {verify.column_names}')
print(f'Sum of token_count: {sum(verify.column("token_count").to_numpy()):,}')

print('\nPreview of first 3 records:')
for i in range(3):
    row = verify.slice(i, 1).to_pylist()[0]
    print(f'\n── Record {i+1} ──')
    print(f'  dump  : {row["dump"]}')
    print(f'  url   : {row["url"]}')
    print(f'  tokens: {row["token_count"]}')
    print(f'  text  : {row["text"][:150]}...')

输出文件总行数 : 1,749,277
输出文件列名   : ['text', 'id', 'dump', 'url', 'date', 'file_path', 'language', 'language_score', 'token_count', 'index']
token_count 总和: 1,000,117,633

前 3 条记录预览：

── 记录 1 ──
  dump  : CC-MAIN-2013-20
  url   : http://6shooter.co.nz/
  tokens: 126
  text  : Times Tables = Fun
Stripes - Cosmetic Bag
3 in 1 Plastic Fruit Vegetable Skin Rotary Blade Rotating Peeler
Women's Weekly Smart Food Cookbook
Veggie M...

── 记录 2 ──
  dump  : CC-MAIN-2013-20
  url   : http://blog.tepapa.govt.nz/tag/gondwana/
  tokens: 253
  text  : I was walking along the corridor at the back of Te Papa the other day and spotted these boxes….
You see some quite strange things out the back of Te P...

── 记录 3 ──
  dump  : CC-MAIN-2013-20
  url   : http://fire.org.nz/About-Us/History/Pages/2000s.aspx
  tokens: 318
  text  : The long running industrial dispute was resolved in June 2001 when a collective agreement was signed with the Professional Firefighters Union.
Firewis...


## Speed Estimate

| Stage | Estimate |
|---|---|
| Cell 2-5 scan + quota + file selection | < 5 sec |
| **Cell 6 main sampling** | **1-3 min** |
| Cell 7-8 output stats + verification | < 5 sec |
| **Total** | **≈ 1.5-3.5 min** |

**Why it's so fast:**
- Parquet is columnar — read the `token_count` column directly and accumulate, **completely skipping tiktoken**
- Each file is read in one shot (pyarrow's C++ implementation)
- Stops as soon as the target is hit — likely only ~100 files processed

## Output Files
- `nz_1B_sample.parquet` — the sample (Snappy compressed, ~3-5 GB)
- `sampling_stats.json` — sampling statistics for each CC-MAIN batch

## Description in the paper
> *We employ stratified random sampling across Common Crawl batches (CC-MAIN-YYYY-WW), allocating at least one file per batch to ensure temporal coverage. The number of files sampled per batch is proportional to the batch's file count. Token counts are taken directly from the pre-computed `token_count` field. Random seed = 42 for reproducibility.*